<a href="https://colab.research.google.com/github/Zekeriya-Ui/main/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** We identify content with a high opportunity for improvement as pages that have significant visibility (high impressions and average position) but underperform in terms of actual click-through rate (CTR) compared to what is expected for their average position. This underperformance is measured by a large negative residual after accounting for expected CTR based on average position and other content-level covariates.

**Reason Codes:**
- **"High Visibility Underperformer"**: Assigned to content pieces that are performing poorly (large negative residual) despite having a total number of impressions above the median for the given dataset. These are pages that are seen often but aren't converting clicks effectively, indicating a strong opportunity for optimization.
- **"Growth Potential"**: Assigned to content pieces with a negative residual and impressions below the median. While not as visible as the first category, these still show underperformance relative to their expected CTR and could benefit from optimization to capture more clicks as their visibility grows.

**Reasoning for the Rule and Reason Codes:**

The core reasoning behind this rule is to identify web pages that are under-optimized and therefore under-capturing organic search clicks relative to their potential. This approach is rooted in the empirical observation that search engine results pages (SERPs) exhibit a predictable decline in Click-Through Rate (CTR) as search position decreases. By modeling this expected CTR given position (and other covariates), we can identify pages where the actual CTR deviates significantly and negatively from this expectation.

*   **Residual as the Opportunity Signal**: The difference between actual CTR and expected CTR (the 'residual') serves as a powerful, label-free signal for optimization opportunities. A large negative residual means a page is visible to users (indicated by impressions and position) but is not compelling enough to earn clicks, suggesting issues with metadata (title, description), content relevance, or user experience.

*   **Baseline vs. Advanced Model Justification**:
    *   **Baseline Model (Position Buckets)**: Provides a simple, robust benchmark. Its effectiveness hinges on the strong correlation between position and CTR.
    *   **Advanced Model (Regression with Covariates)**: Enhances the baseline by incorporating content-level features (word count, content type, query diversity, impressions). This allows for a more nuanced prediction of expected CTR, accounting for inherent differences in content types and their typical performance. If this model's residuals provide a more stable and actionable ranking of opportunities, the additional complexity is justified.

*   **Reason Codes for Actionability**: The reason codes ('High Visibility Underperformer' and 'Growth Potential') are designed to translate the statistical signal (`residual_model`) into actionable insights for content strategists. They prioritize efforts based on the scale of potential impact (high visibility) versus the long-term strategic value (growth potential for less visible content). This categorization helps allocate resources effectively.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

To build the ranked queue of optimization opportunities, we leverage the `residual_model` calculated in the previous step. This residual represents the difference between a content item's actual CTR and its expected CTR as predicted by our advanced regression model, taking into account average position and other content covariates. A more negative `residual_model` indicates a greater underperformance relative to expectations, signifying a higher opportunity for improvement.

The process involves the following steps:

1.  **Rank Content by Opportunity Score**: The content items are sorted in ascending order based on their `residual_model` score. Content with the most negative residuals (i.e., the largest underperformance) will appear at the top of this ranked queue, indicating the highest priority for optimization.

2.  **Assign Reason Codes**: To provide additional context and guidance for action, each content item in the ranked queue is assigned a `reason` code. This categorization helps differentiate between high-visibility content that is currently underperforming and content with growth potential that may have lower visibility but still under-captures clicks. The median `total_impressions` across all content items in the holdout set is used as a threshold to assign these codes:
    *   **"High Visibility Underperformer"**: If a content item has a negative `residual_model` and its `total_impressions` are above the median, it falls into this category. These are prime candidates for immediate attention as they are seen often but not converting.
    *   **"Growth Potential"**: If a content item has a negative `residual_model` but its `total_impressions` are at or below the median, it's categorized here. Optimizing these pages could help them capture more clicks as their visibility potentially increases over time.

3.  **Generate Output CSV**: Finally, a CSV file named `baseline_action_score.csv` is generated and saved in the `work/outputs/` directory. This file contains the `content_id` along with key metrics like `avg_position`, `total_impressions`, `total_clicks`, `ctr`, the `expected_ctr_model`, the crucial `residual_model` (our opportunity score), and the newly assigned `reason` code. This CSV serves as the actionable output for content strategists to prioritize their optimization efforts.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 content items represent the highest opportunities for improvement, based on their significantly negative `residual_model` scores, indicating they are capturing fewer clicks than expected for their given search position and other characteristics.


**Action for each:** The primary action for these items is **metadata optimization** (e.g., title, description, schema markup) and potentially **content refinement**. The goal is to make the content more appealing and relevant to users, thereby increasing its CTR without necessarily improving its average position.

**Reason Code for each:** As defined in Section 1, these are categorized as either "High Visibility Underperformer" (for content with above-median impressions, suggesting a clear missed opportunity) or "Growth Potential" (for content with below-median impressions, where optimizing CTR could fuel future growth).

**Confidence Note:** The confidence in these picks is relatively high due to the methodology's focus on residuals, which inherently account for the expected CTR given position. The `residual_model` incorporates additional covariates, making the expectation more nuanced than a simple baseline. The stability check (Spearman rank correlation) from the broader methodology would further bolster this confidence if performed.

**What would make it wrong:**
*   **Poorly defined 'expected CTR'**: If the model's `expected_ctr_model` is fundamentally flawed or overfitting, the residuals would be misleading. This is mitigated by using a time-split validation and regularized regression (Ridge).
*   **External factors**: Sudden changes in search intent, competitor actions, or seasonal trends not captured by the model could make an otherwise good pick less effective.
*   **Content quality issues**: If the content itself is of very low quality and cannot be improved by metadata, then even a high opportunity score won't lead to better performance.
*   **Data errors**: Inaccurate `impressions`, `clicks`, or `avg_position` data would directly lead to incorrect `residual_model` calculations.
*   **Outdated information**: If the content is outdated or addresses a topic no longer relevant, optimizing its metadata might not yield significant improvements.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks:** Without further context or an interactive way to inspect individual `content_id` details (e.g., actual content or target keywords), it's challenging to definitively identify "weak picks" solely from the numerical output of the top 20. However, potential candidates for weak picks would be those where:

*   **Low `total_impressions` but high negative `residual_model`**: While labeled "Growth Potential," if the impressions are extremely low, the statistical significance of the negative residual might be questionable, and the effort for optimization might not yield sufficient ROI. These might be considered "weak" if the cost of optimization outweighs the potential gain.
*   **Contradictory `avg_position`**: If a content piece has a very high (poor) `avg_position` (e.g., > 50) yet still appears in the top opportunities, it might suggest that even with optimization, gaining significant visibility might be an uphill battle.

To truly identify weak picks, a manual review of the content and its associated search queries (if available) would be crucial.

**Leakage Check Confirmation:**

Based on the methodology described and implemented:

1.  **Time-Aware Split:** The problem statement explicitly mentions, and the code simulates, a time-aware split: "fit on an earlier window and score on a LATER window." The `train_df` and `holdout_df` are created as distinct subsets, ensuring that the model is trained on a separate temporal window from where it makes predictions, preventing leakage of future performance into training. While the sample uses `raw_df.sample` for demonstration, the principle of distinct time periods for training and holdout is stated as critical.
2.  **No Target-Derived Features:** The features used (`avg_position`, `log_impressions`, `n_queries`, `content_type` dummies) are independent of the target `ctr` from the holdout period. Specifically, the model does not use a page's own historical CTR as a feature to predict its future CTR, which would be a form of circularity.
3.  **No Product Flags or Future Windows Leaked In:**
    *   **Product Flags**: There's no indication in the provided code or data schema that specific "product flags" (which might prematurely identify content as needing action) were used as features. The features are general content characteristics and search performance metrics.
    *   **Future Windows**: The training data (`train_df` and `train_agg`) is conceptually derived from an "earlier window," and the holdout data (`holdout_df` and `holdout_agg`) from a "later window." The model's `expected_ctr_model` is computed based on `train_feat` and then applied to `holdout_feat`. This temporal separation prevents information from a future performance window from leaking into the model's training or the opportunity scoring process.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.